# Ábacos de Venturini — Flexão Composta Reta  
## Versão com dataset validado e interpolação automática em duas etapas

Este notebook organiza o dimensionamento preliminar de seções retangulares de concreto armado submetidas à **flexão composta reta**, com base na lógica de uso dos **ábacos de Venturini**.

A sequência adotada é:

1. entrada de dados;
2. cálculo de `ν` e `μ`;
3. leitura do dataset validado dos ábacos;
4. escolha do ábaco ou interpolação entre ábacos;
5. obtenção automática de `ω`;
6. cálculo da área de aço `As`;
7. verificações normativas básicas;
8. sugestão preliminar de detalhamento;
9. geração de memorial de cálculo.

> **Importante:** esta versão não usa interpolação 3D direta como método principal.  
> A obtenção de `ω` segue a lógica tecnicamente recomendada:
>
> 1. interpolação 2D dentro de cada ábaco, usando `(ν, μ)`;
> 2. interpolação linear entre ábacos vizinhos, usando `d'/h`.
>
> O dataset utilizado deve possuir pontos digitalizados, saneados e validados, sem duplicidades conflitantes de `ω`.


## 1. Convenções, hipóteses e fórmulas principais

### Convenções adotadas

- `Nd` positivo: compressão.
- `Md` informado em módulo, em `kNm`.
- dimensões em `cm`;
- tensões convertidas para `kN/cm²`;
- seção retangular;
- armadura simétrica nas duas faces principais;
- flexão composta reta em uma direção;
- cálculo preliminar, exigindo conferência final do engenheiro responsável.

### Parâmetros adimensionais

\[
\nu = \frac{N_d}{A_c \cdot f_{cd}}
\]

\[
\mu = \frac{M_d}{A_c \cdot h \cdot f_{cd}}
\]

Como `Md` é informado em `kNm`, o notebook converte para `kN.cm` multiplicando por 100.

### Área de aço a partir de `ω`

\[
A_s = \frac{\omega \cdot A_c \cdot f_{cd}}{f_{yd}}
\]

### Observação crítica

Os ábacos de Venturini já embutem hipóteses de equilíbrio, compatibilidade de deformações, domínios de deformação e comportamento dos materiais.  
Por isso, substituir o ábaco por uma expressão simplificada sem validação pode gerar erro estrutural relevante.

In [ ]:
# ============================================================
# 2. IMPORTAÇÕES E CONFIGURAÇÕES GERAIS
# ============================================================

from dataclasses import dataclass, asdict
from datetime import datetime
import math
import os
import warnings

import numpy as np
import pandas as pd
from scipy.interpolate import LinearNDInterpolator

try:
    from IPython.display import display, Markdown
except Exception:
    display = None
    Markdown = None

# Impressão tabular mais legível
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

# Níveis usuais dos ábacos de Venturini para d'/h.
# Esta lista será atualizada automaticamente após a leitura do CSV validado.
ABACOS_DH_DISPONIVEIS = [0.05, 0.10, 0.15, 0.20, 0.25]

print("Ambiente preparado.")


## 3. Entrada de dados

Edite os valores da célula abaixo.

Recomendações:

- use dimensões reais da seção em `cm`;
- informe `d'` como cobrimento mecânico, isto é, distância do centroide da armadura à face comprimida/tracionada correspondente;
- use esforços de cálculo, não esforços característicos;
- registre a origem dos esforços: combinação, modelo estrutural, envoltória ou relatório de cálculo.

In [ ]:
# ============================================================
# 3. ENTRADA DE DADOS
# ============================================================
# Exemplo didático. Substitua pelos dados do seu caso real.

DADOS_ENTRADA = {
    # Geometria
    "b_cm": 20.0,             # largura da seção
    "h_cm": 50.0,             # altura da seção na direção da flexão
    "d_linha_cm": 5.0,        # d' = cobrimento mecânico até o centro da armadura

    # Materiais
    "fck_mpa": 30.0,          # resistência característica do concreto
    "fyk_mpa": 500.0,         # aço CA-50 usual
    "gamma_c": 1.40,          # coeficiente de ponderação do concreto
    "gamma_s": 1.15,          # coeficiente de ponderação do aço
    "alpha_cc": 1.00,         # mantido como 1,00 para compatibilidade com o cálculo clássico fcd=fck/gamma_c

    # Esforços de cálculo
    "Nd_kN": 900.0,           # compressão positiva
    "Md_kNm": 120.0,          # momento de cálculo em módulo

    # Informações para rastreabilidade
    "identificacao_secao": "Pilar P1 - exemplo didático",
    "origem_esforcos": "Exemplo interno do notebook. Substituir por combinação/modelo estrutural real.",
    "observacoes": "Cálculo preliminar com armadura simétrica."
}

DADOS_ENTRADA

In [ ]:
# ============================================================
# 4. ESTRUTURAS DE DADOS E VALIDAÇÕES
# ============================================================

@dataclass
class DadosSecao:
    b_cm: float
    h_cm: float
    d_linha_cm: float
    fck_mpa: float
    fyk_mpa: float
    gamma_c: float
    gamma_s: float
    alpha_cc: float
    Nd_kN: float
    Md_kNm: float
    identificacao_secao: str = ""
    origem_esforcos: str = ""
    observacoes: str = ""

def validar_dados(d: DadosSecao) -> list:
    """
    Valida dados geométricos, materiais e esforços.
    Retorna uma lista de alertas técnicos.
    """
    alertas = []

    if d.b_cm <= 0:
        raise ValueError("A largura b deve ser positiva.")
    if d.h_cm <= 0:
        raise ValueError("A altura h deve ser positiva.")
    if d.d_linha_cm <= 0:
        raise ValueError("O cobrimento mecânico d' deve ser positivo.")
    if d.d_linha_cm >= d.h_cm / 2:
        raise ValueError("d' deve ser menor que h/2 para armadura simétrica em duas faces.")

    if d.fck_mpa <= 0:
        raise ValueError("fck deve ser positivo.")
    if d.fyk_mpa <= 0:
        raise ValueError("fyk deve ser positivo.")
    if d.gamma_c <= 1.0:
        alertas.append("gamma_c menor ou igual a 1,0. Conferir coeficiente de ponderação.")
    if d.gamma_s <= 1.0:
        alertas.append("gamma_s menor ou igual a 1,0. Conferir coeficiente de ponderação.")
    if d.alpha_cc <= 0:
        raise ValueError("alpha_cc deve ser positivo.")

    if d.Nd_kN < 0:
        alertas.append("Nd negativo: caso de tração. Verifique se os ábacos/rotina são aplicáveis ao caso.")
    if d.Md_kNm < 0:
        alertas.append("Md negativo informado. O notebook usará o módulo para consulta ao ábaco.")

    rel_dh = d.d_linha_cm / d.h_cm
    if rel_dh < min(ABACOS_DH_DISPONIVEIS) or rel_dh > max(ABACOS_DH_DISPONIVEIS):
        alertas.append(
            f"A relação d'/h = {rel_dh:.3f} está fora da faixa dos ábacos cadastrados "
            f"({min(ABACOS_DH_DISPONIVEIS):.2f} a {max(ABACOS_DH_DISPONIVEIS):.2f}). "
            "Evite extrapolação sem validação."
        )

    return alertas

dados = DadosSecao(**DADOS_ENTRADA)
alertas_entrada = validar_dados(dados)

print("Dados validados.")
if alertas_entrada:
    print("\nAlertas:")
    for a in alertas_entrada:
        print(" -", a)

## 4. Cálculo dos parâmetros geométricos, materiais e adimensionais

Esta etapa calcula:

- área bruta de concreto `Ac`;
- resistência de cálculo do concreto `fcd`;
- resistência de cálculo do aço `fyd`;
- relação `d'/h`;
- parâmetros `ν` e `μ`.

Esses valores são a entrada para escolha/leitura/interpolação dos ábacos.

In [ ]:
# ============================================================
# 5. CÁLCULO DOS PARÂMETROS ADIMENSIONAIS
# ============================================================

def calcular_parametros_basicos(d: DadosSecao) -> dict:
    Ac_cm2 = d.b_cm * d.h_cm
    fcd_kN_cm2 = (d.alpha_cc * d.fck_mpa / d.gamma_c) / 10.0
    fyd_kN_cm2 = (d.fyk_mpa / d.gamma_s) / 10.0
    rel_dh = d.d_linha_cm / d.h_cm

    Md_kNcm = abs(d.Md_kNm) * 100.0

    nu = d.Nd_kN / (Ac_cm2 * fcd_kN_cm2)
    mu = Md_kNcm / (Ac_cm2 * d.h_cm * fcd_kN_cm2)

    return {
        "Ac_cm2": Ac_cm2,
        "fcd_kN_cm2": fcd_kN_cm2,
        "fyd_kN_cm2": fyd_kN_cm2,
        "rel_dh": rel_dh,
        "Md_kNcm": Md_kNcm,
        "nu": nu,
        "mu": mu,
    }

param = calcular_parametros_basicos(dados)

tabela_parametros = pd.DataFrame([
    ["Ac", param["Ac_cm2"], "cm²"],
    ["fcd", param["fcd_kN_cm2"], "kN/cm²"],
    ["fyd", param["fyd_kN_cm2"], "kN/cm²"],
    ["d'/h", param["rel_dh"], "-"],
    ["ν", param["nu"], "-"],
    ["μ", param["mu"], "-"],
], columns=["Parâmetro", "Valor", "Unidade"])

tabela_parametros

In [ ]:
# ============================================================
# 6. ESCOLHA DO ÁBACO / INTERPOLAÇÃO ENTRE ÁBACOS d'/h
# ============================================================
# Recomendação técnica implementada:
# 1) interpolar ω dentro de cada ábaco em função de (ν, μ);
# 2) interpolar linearmente entre os ábacos vizinhos em função de d'/h;
# 3) NÃO usar interpolação 3D direta como método principal.

CAMINHO_ABACOS_CSV = "dataset_venturini_final_sem_extremos_duplicados.csv"

COLUNAS_OBRIGATORIAS_DATASET = [
    "abaco_id",
    "d_linha_h",
    "nu",
    "mu",
    "omega",
    "fyk",
    "tipo_armadura",
    "status_validacao",
]

def resolver_caminho_csv(caminho_csv: str) -> str:
    """
    Resolve o caminho do CSV em ambientes diferentes:
    - Colab: /content/
    - execução local: diretório atual
    - sandbox/ambiente de teste: /mnt/data/
    """
    candidatos = [
        caminho_csv,
        os.path.join(os.getcwd(), caminho_csv),
        os.path.join("/content", os.path.basename(caminho_csv)),
        os.path.join("/mnt/data", os.path.basename(caminho_csv)),
        os.path.join("/mnt/data", "dataset_venturini_final_sem_extremos_duplicados(1).csv"),
        os.path.join("/mnt/data", "dataset_venturini_final_sem_extremos_duplicados (1).csv"),
    ]

    for candidato in candidatos:
        if os.path.exists(candidato):
            return candidato

    raise FileNotFoundError(
        "CSV dos ábacos não encontrado. Envie o arquivo "
        "'dataset_venturini_final_sem_extremos_duplicados.csv' para o ambiente "
        "do Colab ou ajuste CAMINHO_ABACOS_CSV."
    )

def carregar_dataset_venturini(caminho_csv):
    """
    Carrega, valida e saneia o dataset digitalizado dos ábacos de Venturini.

    Estrutura esperada:
    abaco_id, d_linha_h, nu, mu, omega, fyk, tipo_armadura,
    fonte, pagina, curva_id, ponto_id, digitizador, revisor,
    status_validacao, observacao.
    """
    caminho_resolvido = resolver_caminho_csv(caminho_csv)
    df = pd.read_csv(caminho_resolvido)

    faltantes = [c for c in COLUNAS_OBRIGATORIAS_DATASET if c not in df.columns]
    if faltantes:
        raise ValueError(f"Colunas obrigatórias ausentes: {faltantes}")

    for col in ["d_linha_h", "nu", "mu", "omega"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    if df[["d_linha_h", "nu", "mu", "omega"]].isna().any().any():
        raise ValueError("Há valores não numéricos ou inválidos na base.")

    df["status_validacao"] = df["status_validacao"].astype(str).str.lower().str.strip()
    df = df[df["status_validacao"] == "validado"].copy()

    if df.empty:
        raise ValueError("Não há pontos com status_validacao = 'validado'.")

    duplicados = (
        df.groupby(["d_linha_h", "nu", "mu"])["omega"]
        .nunique()
        .reset_index(name="n_omega")
    )

    conflitos = duplicados[duplicados["n_omega"] > 1]

    if not conflitos.empty:
        raise ValueError("Há coordenadas com múltiplos valores de omega.")

    df.attrs["caminho_csv"] = caminho_resolvido
    return df

def montar_interpoladores_por_abaco(df):
    """
    Monta um interpolador linear 2D para cada ábaco d'/h.
    Entrada de cada interpolador: (ν, μ).
    Saída: ω.
    """
    interpoladores = {}

    for d_h, grupo in df.groupby("d_linha_h"):
        pontos = grupo[["nu", "mu"]].to_numpy()
        valores = grupo["omega"].to_numpy()

        if len(grupo) < 3:
            raise ValueError(f"Poucos pontos para montar interpolador em d'/h = {d_h}.")

        interpolador = LinearNDInterpolator(pontos, valores)
        interpoladores[float(d_h)] = interpolador

    return interpoladores

def interpolar_omega_venturini(df, nu, mu, d_linha_h):
    """
    Interpola ω conforme a lógica dos ábacos:

    1. Se d'/h coincidir com um ábaco existente:
       interpolação 2D em (ν, μ) dentro desse ábaco.

    2. Se d'/h estiver entre dois ábacos:
       calcula ω nos dois ábacos vizinhos por interpolação 2D
       e interpola linearmente em d'/h.

    3. Se d'/h estiver fora da faixa digitalizada:
       bloqueia o cálculo. Não extrapola automaticamente.
    """
    interpoladores = montar_interpoladores_por_abaco(df)
    dhs = sorted(interpoladores.keys())

    if d_linha_h < min(dhs) or d_linha_h > max(dhs):
        raise ValueError(
            f"d'/h = {d_linha_h:.4f} fora da faixa digitalizada "
            f"[{min(dhs):.2f}, {max(dhs):.2f}]. Extrapolação não permitida."
        )

    # Caso coincida exatamente com um ábaco existente
    for dh in dhs:
        if math.isclose(d_linha_h, dh, rel_tol=1e-12, abs_tol=1e-12):
            omega = interpoladores[dh](nu, mu)

            if np.isnan(omega):
                raise ValueError(
                    "O ponto (ν, μ) está fora da região interpolável do ábaco selecionado."
                )

            return {
                "omega": float(omega),
                "dh_inferior": float(dh),
                "dh_superior": float(dh),
                "metodo": "interpolacao_linear_2d_no_abaco",
                "alerta": "",
            }

    # Caso esteja entre dois ábacos
    dh_inf = max(d for d in dhs if d <= d_linha_h)
    dh_sup = min(d for d in dhs if d >= d_linha_h)

    omega_inf = interpoladores[dh_inf](nu, mu)
    omega_sup = interpoladores[dh_sup](nu, mu)

    if np.isnan(omega_inf) or np.isnan(omega_sup):
        raise ValueError(
            "O ponto (ν, μ) está fora da região interpolável em um dos ábacos vizinhos."
        )

    omega = omega_inf + (omega_sup - omega_inf) * (
        (d_linha_h - dh_inf) / (dh_sup - dh_inf)
    )

    return {
        "omega": float(omega),
        "dh_inferior": float(dh_inf),
        "dh_superior": float(dh_sup),
        "metodo": "interpolacao_2d_em_nu_mu_mais_interpolacao_linear_em_dh",
        "alerta": "",
    }

def selecionar_abacos_dh(rel_dh: float, niveis=ABACOS_DH_DISPONIVEIS) -> dict:
    """
    Seleciona o ábaco mais próximo e, quando aplicável, os dois ábacos
    que envolvem a relação d'/h para interpolação.
    """
    niveis = sorted(float(x) for x in niveis)
    mais_proximo = min(niveis, key=lambda x: abs(x - rel_dh))

    if rel_dh <= niveis[0]:
        inferior = superior = niveis[0]
        situacao = "abaixo_da_faixa" if rel_dh < niveis[0] else "exato"
    elif rel_dh >= niveis[-1]:
        inferior = superior = niveis[-1]
        situacao = "acima_da_faixa" if rel_dh > niveis[-1] else "exato"
    else:
        inferior = max(x for x in niveis if x <= rel_dh)
        superior = min(x for x in niveis if x >= rel_dh)
        situacao = "exato" if math.isclose(inferior, superior) else "interpolar"

    return {
        "rel_dh": rel_dh,
        "abaco_mais_proximo": mais_proximo,
        "abaco_inferior": inferior,
        "abaco_superior": superior,
        "situacao": situacao,
    }

# Leitura do dataset validado
df_abacos_venturini = carregar_dataset_venturini(CAMINHO_ABACOS_CSV)
ABACOS_DH_DISPONIVEIS = sorted(df_abacos_venturini["d_linha_h"].unique())

selecao_abaco = selecionar_abacos_dh(param["rel_dh"], ABACOS_DH_DISPONIVEIS)

print("Dataset dos ábacos carregado:")
print(f" - arquivo: {df_abacos_venturini.attrs.get('caminho_csv')}")
print(f" - pontos válidos: {len(df_abacos_venturini)}")
print(f" - ábacos d'/h disponíveis: {ABACOS_DH_DISPONIVEIS}")

print("\nEscolha do ábaco:")
for k, v in selecao_abaco.items():
    print(f" - {k}: {v}")

if selecao_abaco["situacao"] in ["abaixo_da_faixa", "acima_da_faixa"]:
    print("\nALERTA: relação d'/h fora da faixa dos ábacos cadastrados. Extrapolação não recomendada e bloqueada no cálculo de ω.")
elif selecao_abaco["situacao"] == "interpolar":
    print("\nUsar interpolação em duas etapas: (ν, μ) dentro dos ábacos vizinhos e depois interpolação linear em d'/h.")
else:
    print("\nUsar diretamente o ábaco correspondente, com interpolação 2D em (ν, μ).")


## 5. Obtenção de `ω`

Há dois modos tecnicamente aceitáveis nesta versão:

### Modo 1 — `manual`

Você lê `ω` diretamente no ábaco de Venturini e informa o valor no notebook.  
Esse modo é simples, rastreável e adequado quando os ábacos estão em PDF/livro.

### Modo 2 — `csv_interpolado`

Você fornece a base digitalizada, saneada e validada dos ábacos, no formato:

| coluna | descrição |
|---|---|
| `abaco_id` | identificação do ábaco, por exemplo A-1 a A-5 |
| `d_linha_h` | relação `d'/h` do ábaco |
| `nu` | coordenada adimensional `ν` |
| `mu` | coordenada adimensional `μ` |
| `omega` | valor da curva `ω` |
| `fyk` | aço de referência |
| `tipo_armadura` | arranjo de armadura representado pelo ábaco |
| `status_validacao` | deve estar como `validado` |
| demais campos | metadados de fonte, página, curva, ponto, digitizador e revisor |

> Nesta versão, o modo CSV usa **interpolação em duas etapas**:
>
> 1. interpolação 2D dentro do ábaco, com entrada `(ν, μ)` e saída `ω`;
> 2. interpolação linear entre ábacos vizinhos, quando `d'/h` estiver entre dois valores cadastrados.
>
> A interpolação 3D direta `(d'/h, ν, μ) → ω` não é usada como método principal.


In [ ]:
# ============================================================
# 7. CRIAÇÃO DE TEMPLATE PARA BASE DIGITALIZADA DOS ÁBACOS
# ============================================================

def criar_template_csv_abacos(caminho="venturini_abacos_template_15_colunas.csv"):
    """
    Cria um arquivo CSV vazio com a estrutura recomendada para digitalização,
    validação e rastreabilidade dos ábacos de Venturini.
    """
    colunas = [
        "abaco_id",
        "d_linha_h",
        "nu",
        "mu",
        "omega",
        "fyk",
        "tipo_armadura",
        "fonte",
        "pagina",
        "curva_id",
        "ponto_id",
        "digitizador",
        "revisor",
        "status_validacao",
        "observacao",
    ]
    df_template = pd.DataFrame(columns=colunas)
    df_template.to_csv(caminho, index=False, encoding="utf-8")
    return caminho

template_path = criar_template_csv_abacos()
print(f"Template criado: {template_path}")
print("Use este template apenas para novas digitalizações. O cálculo atual já usa o dataset validado informado em CAMINHO_ABACOS_CSV.")


In [ ]:
# ============================================================
# 8. DIAGNÓSTICO DO DATASET DIGITALIZADO DOS ÁBACOS
# ============================================================

def diagnosticar_dataset_venturini(df: pd.DataFrame) -> dict:
    """
    Gera diagnóstico sintético do dataset carregado.
    """
    resumo = {
        "linhas": int(len(df)),
        "colunas": int(len(df.columns)),
        "abacos": sorted(df["abaco_id"].astype(str).unique().tolist()),
        "d_linha_h": sorted(float(x) for x in df["d_linha_h"].unique()),
        "omega_min": float(df["omega"].min()),
        "omega_max": float(df["omega"].max()),
        "nu_min": float(df["nu"].min()),
        "nu_max": float(df["nu"].max()),
        "mu_min": float(df["mu"].min()),
        "mu_max": float(df["mu"].max()),
    }

    conflitos = (
        df.groupby(["d_linha_h", "nu", "mu"])["omega"]
        .nunique()
        .reset_index(name="n_omega")
    )
    conflitos = conflitos[conflitos["n_omega"] > 1]
    resumo["coordenadas_com_omega_conflitante"] = int(len(conflitos))

    por_abaco = (
        df.groupby(["abaco_id", "d_linha_h"])
        .agg(
            pontos=("omega", "size"),
            curvas=("omega", "nunique"),
            nu_min=("nu", "min"),
            nu_max=("nu", "max"),
            mu_min=("mu", "min"),
            mu_max=("mu", "max"),
        )
        .reset_index()
        .sort_values(["d_linha_h", "abaco_id"])
    )

    return resumo, por_abaco

resumo_dataset, tabela_por_abaco = diagnosticar_dataset_venturini(df_abacos_venturini)

print("Diagnóstico do dataset:")
for k, v in resumo_dataset.items():
    print(f" - {k}: {v}")

print("\nResumo por ábaco:")
display(tabela_por_abaco) if display else print(tabela_por_abaco)


In [ ]:
# ============================================================
# 9. DEFINIÇÃO DO MODO DE OBTENÇÃO DO OMEGA
# ============================================================
# Opções:
# - "manual": informe OMEGA_LIDO a partir da leitura do ábaco.
# - "csv_interpolado": usa o dataset validado e interpolação em duas etapas.

MODO_OMEGA = "csv_interpolado"

# Modo manual: substitua pelo valor efetivamente lido no ábaco, se usar MODO_OMEGA = "manual".
OMEGA_LIDO = 0.30

# Rastreabilidade da leitura manual
FONTE_OMEGA_MANUAL = {
    "abaco": f"d'/h ≈ {selecao_abaco['abaco_mais_proximo']:.2f}",
    "pagina_ou_figura": "Informar página/figura do material de Venturini",
    "responsavel_leitura": "Informar responsável",
    "data_leitura": datetime.now().strftime("%Y-%m-%d"),
    "observacao": "Valor didático. Substituir pelo valor lido/interpolado no caso real."
}

def obter_omega(modo: str) -> dict:
    modo = modo.lower().strip()

    if modo == "manual":
        if OMEGA_LIDO is None:
            raise ValueError("OMEGA_LIDO não foi informado.")
        if OMEGA_LIDO < 0:
            raise ValueError("OMEGA_LIDO não pode ser negativo.")

        return {
            "omega": float(OMEGA_LIDO),
            "modo": "manual",
            "metodo": "leitura_manual_do_abaco",
            "fonte": FONTE_OMEGA_MANUAL,
            "alertas": []
        }

    if modo == "csv_interpolado":
        r = interpolar_omega_venturini(
            df=df_abacos_venturini,
            nu=param["nu"],
            mu=param["mu"],
            d_linha_h=param["rel_dh"],
        )

        alertas = []
        if r.get("alerta"):
            alertas.append(r["alerta"])

        return {
            "omega": r["omega"],
            "modo": "csv_interpolado",
            "metodo": r["metodo"],
            "fonte": {
                "arquivo": df_abacos_venturini.attrs.get("caminho_csv", CAMINHO_ABACOS_CSV),
                "dh_inferior": r["dh_inferior"],
                "dh_superior": r["dh_superior"],
                "observacao": "Interpolação em duas etapas: 2D em (ν, μ) dentro dos ábacos e linear em d'/h entre ábacos."
            },
            "alertas": alertas
        }

    raise ValueError("MODO_OMEGA deve ser 'manual' ou 'csv_interpolado'.")

resultado_omega = obter_omega(MODO_OMEGA)

print("Resultado para omega:")
print(f" - modo: {resultado_omega['modo']}")
print(f" - método: {resultado_omega['metodo']}")
print(f" - omega: {resultado_omega['omega']:.4f}")

print("\nFonte / rastreabilidade:")
for k, v in resultado_omega["fonte"].items():
    print(f" - {k}: {v}")

if resultado_omega["alertas"]:
    print("\nAlertas:")
    for a in resultado_omega["alertas"]:
        print(" -", a)


## 6. Cálculo da área de aço e verificações básicas

A área de aço calculada pelo ábaco é comparada com limites preliminares usuais para armadura longitudinal.

Nesta versão, são feitas verificações automáticas básicas:

- `As` calculada por `ω`;
- armadura mínima preliminar;
- armadura máxima preliminar;
- momento mínimo associado à excentricidade mínima;
- alertas de domínio de uso e rastreabilidade.

> Atenção: esta etapa não substitui a verificação completa de pilar, efeitos locais de segunda ordem, índice de esbeltez, imperfeições geométricas, combinações normativas e detalhamento final.

In [ ]:
# ============================================================
# 10. CÁLCULO DE As E VERIFICAÇÕES BÁSICAS
# ============================================================

def calcular_area_aco_e_verificacoes(d: DadosSecao, param: dict, omega: float) -> dict:
    Ac = param["Ac_cm2"]
    fcd = param["fcd_kN_cm2"]
    fyd = param["fyd_kN_cm2"]

    As_calc = omega * Ac * fcd / fyd

    # Verificações preliminares usuais para elementos comprimidos.
    # Conferir sempre a edição normativa e as condições específicas do projeto.
    As_min_por_N = 0.15 * max(d.Nd_kN, 0.0) / fyd
    As_min_por_Ac = 0.004 * Ac
    As_min = max(As_min_por_N, As_min_por_Ac)

    As_max = 0.08 * Ac

    As_req = max(As_calc, As_min)

    # Excentricidade mínima preliminar: e_min = max(h/30, 2 cm)
    e_min_cm = max(d.h_cm / 30.0, 2.0)
    Md_min_kNm = max(d.Nd_kN, 0.0) * e_min_cm / 100.0

    alertas = []

    if abs(d.Md_kNm) < Md_min_kNm and d.Nd_kN > 0:
        alertas.append(
            f"Md informado ({abs(d.Md_kNm):.2f} kNm) é menor que o momento mínimo preliminar "
            f"({Md_min_kNm:.2f} kNm). Conferir excentricidade mínima e combinações."
        )

    if As_calc < As_min:
        alertas.append(
            f"As calculada pelo ábaco ({As_calc:.2f} cm²) é menor que As_min ({As_min:.2f} cm²). "
            "Adotar pelo menos As_min, salvo justificativa normativa específica."
        )

    if As_req > As_max:
        alertas.append(
            f"As requerida ({As_req:.2f} cm²) excede As_max preliminar ({As_max:.2f} cm²). "
            "Rever seção, esforços, materiais ou concepção estrutural."
        )

    if param["nu"] > 1.0:
        alertas.append(
            f"ν = {param['nu']:.3f} elevado. Conferir aplicabilidade do ábaco, compressão no concreto e viabilidade da seção."
        )

    if omega == 0:
        alertas.append("ω = 0. Conferir leitura do ábaco, esforços e domínio do problema.")

    return {
        "As_calc_cm2": As_calc,
        "As_min_por_N_cm2": As_min_por_N,
        "As_min_por_Ac_cm2": As_min_por_Ac,
        "As_min_cm2": As_min,
        "As_max_cm2": As_max,
        "As_req_cm2": As_req,
        "As_por_face_cm2": As_req / 2.0,
        "e_min_cm": e_min_cm,
        "Md_min_kNm": Md_min_kNm,
        "alertas": alertas,
    }

resultado_as = calcular_area_aco_e_verificacoes(dados, param, resultado_omega["omega"])

tabela_as = pd.DataFrame([
    ["ω adotado", resultado_omega["omega"], "-"],
    ["As calculada pelo ábaco", resultado_as["As_calc_cm2"], "cm²"],
    ["As_min por Nd", resultado_as["As_min_por_N_cm2"], "cm²"],
    ["As_min por 0,4% Ac", resultado_as["As_min_por_Ac_cm2"], "cm²"],
    ["As_min adotada", resultado_as["As_min_cm2"], "cm²"],
    ["As requerida", resultado_as["As_req_cm2"], "cm²"],
    ["As por face", resultado_as["As_por_face_cm2"], "cm²"],
    ["As_max preliminar", resultado_as["As_max_cm2"], "cm²"],
    ["e_min preliminar", resultado_as["e_min_cm"], "cm"],
    ["Md_min preliminar", resultado_as["Md_min_kNm"], "kNm"],
], columns=["Item", "Valor", "Unidade"])

display(tabela_as) if display else print(tabela_as)

if resultado_as["alertas"]:
    print("\nAlertas de verificação:")
    for a in resultado_as["alertas"]:
        print(" -", a)
else:
    print("\nSem alertas automáticos nas verificações básicas.")

## 7. Sugestão preliminar de detalhamento da armadura

A célula abaixo sugere combinações de número de barras e diâmetro que atendam à área de aço requerida.

Essa sugestão **não substitui** o detalhamento estrutural completo. Ainda devem ser conferidos:

- número mínimo de barras conforme tipo de seção;
- espaçamento livre entre barras;
- cobrimento nominal;
- diâmetro dos estribos;
- ancoragem;
- emendas;
- interferência com concretagem;
- compatibilidade com a dimensão real da seção.

In [ ]:
# ============================================================
# 11. SUGESTÃO PRELIMINAR DE ARMADURA LONGITUDINAL
# ============================================================

BITOLAS_MM = [8.0, 10.0, 12.5, 16.0, 20.0, 25.0, 32.0]

def area_barra_cm2(phi_mm: float) -> float:
    """Área de uma barra circular em cm²."""
    phi_cm = phi_mm / 10.0
    return math.pi * phi_cm**2 / 4.0

def sugerir_combinacoes_barras(
    As_req_cm2: float,
    As_max_cm2: float,
    bitolas_mm=BITOLAS_MM,
    n_min=4,
    n_max=24,
    apenas_pares=True
) -> pd.DataFrame:
    linhas = []

    for n in range(n_min, n_max + 1):
        if apenas_pares and n % 2 != 0:
            continue

        for phi in bitolas_mm:
            As_barra = area_barra_cm2(phi)
            As_total = n * As_barra

            if As_total >= As_req_cm2 and As_total <= As_max_cm2:
                margem = As_total - As_req_cm2
                linhas.append({
                    "n_barras": n,
                    "diametro_mm": phi,
                    "As_prov_cm2": As_total,
                    "As_req_cm2": As_req_cm2,
                    "margem_cm2": margem,
                    "margem_%": 100.0 * margem / As_req_cm2 if As_req_cm2 > 0 else np.nan,
                    "sugestao": f"{n} Ø {phi:g} mm"
                })

    df = pd.DataFrame(linhas)

    if df.empty:
        return df

    return df.sort_values(["margem_%", "diametro_mm", "n_barras"]).reset_index(drop=True)

df_sugestoes = sugerir_combinacoes_barras(
    As_req_cm2=resultado_as["As_req_cm2"],
    As_max_cm2=resultado_as["As_max_cm2"],
)

if df_sugestoes.empty:
    print("Nenhuma combinação encontrada nos limites configurados. Reveja bitolas, número máximo de barras ou a seção.")
else:
    display(df_sugestoes.head(10)) if display else print(df_sugestoes.head(10))

## 8. Checklist técnico de conformidade

Use este checklist antes de transformar o resultado em memorial definitivo.

### Entrada e esforços

- [ ] esforços são de cálculo;
- [ ] combinação de ações foi identificada;
- [ ] unidade de `Nd` e `Md` foi conferida;
- [ ] sinal de compressão/tração foi conferido;
- [ ] momento mínimo foi considerado quando aplicável.

### Ábacos

- [ ] `d'/h` está dentro da faixa do ábaco;
- [ ] `ν` e `μ` estão dentro do domínio gráfico;
- [ ] leitura de `ω` foi registrada;
- [ ] se houve interpolação, a base digitalizada foi validada;
- [ ] a interpolação foi feita em duas etapas: 2D em `(ν, μ)` e linear em `d'/h`;
- [ ] não houve extrapolação fora da faixa dos ábacos digitalizados.

### Dimensionamento

- [ ] `As` calculada foi comparada com `As_min`;
- [ ] `As` foi comparada com limite máximo;
- [ ] efeitos de segunda ordem foram avaliados;
- [ ] esbeltez do pilar foi verificada;
- [ ] detalhamento atende cobrimento, espaçamento, ancoragem e emendas;
- [ ] o arranjo de barras é construtivamente viável.

### Rastreabilidade

- [ ] memória de cálculo identifica seção, data, responsável e origem dos esforços;
- [ ] parâmetros normativos foram explicitados;
- [ ] hipóteses e limitações foram registradas;
- [ ] resultado foi revisado por engenheiro responsável.

In [ ]:
# ============================================================
# 12. GERAÇÃO DO MEMORIAL DE CÁLCULO EM MARKDOWN
# ============================================================

def formatar_alertas(alertas):
    if not alertas:
        return "- Não foram gerados alertas automáticos nas verificações básicas."
    return "\n".join([f"- {a}" for a in alertas])

def gerar_memorial_markdown(
    d: DadosSecao,
    param: dict,
    selecao_abaco: dict,
    resultado_omega: dict,
    resultado_as: dict,
    df_sugestoes: pd.DataFrame
) -> str:
    data = datetime.now().strftime("%d/%m/%Y %H:%M")

    if df_sugestoes.empty:
        sugestao_txt = "Não foi encontrada combinação automática de barras dentro dos limites configurados."
    else:
        top = df_sugestoes.iloc[0]
        sugestao_txt = (
            f"Sugestão preliminar mais econômica: **{top['sugestao']}**, "
            f"com As_prov = {top['As_prov_cm2']:.2f} cm² "
            f"e margem = {top['margem_%']:.1f}%."
        )

    fonte_omega = resultado_omega.get("fonte", {})

    md = f"""
# Memorial de Cálculo — Flexão Composta Reta com Ábacos de Venturini

**Data de geração:** {data}  
**Seção:** {d.identificacao_secao or "não informada"}  
**Origem dos esforços:** {d.origem_esforcos or "não informada"}  

## 1. Dados de entrada

| Item | Valor |
|---|---:|
| b | {d.b_cm:.2f} cm |
| h | {d.h_cm:.2f} cm |
| d' | {d.d_linha_cm:.2f} cm |
| fck | {d.fck_mpa:.2f} MPa |
| fyk | {d.fyk_mpa:.2f} MPa |
| γc | {d.gamma_c:.3f} |
| γs | {d.gamma_s:.3f} |
| αcc | {d.alpha_cc:.3f} |
| Nd | {d.Nd_kN:.2f} kN |
| Md | {abs(d.Md_kNm):.2f} kNm |

## 2. Parâmetros calculados

| Parâmetro | Valor |
|---|---:|
| Ac | {param['Ac_cm2']:.2f} cm² |
| fcd | {param['fcd_kN_cm2']:.4f} kN/cm² |
| fyd | {param['fyd_kN_cm2']:.4f} kN/cm² |
| d'/h | {param['rel_dh']:.4f} |
| ν | {param['nu']:.4f} |
| μ | {param['mu']:.4f} |

## 3. Seleção do ábaco

| Item | Valor |
|---|---:|
| Ábaco mais próximo | d'/h = {selecao_abaco['abaco_mais_proximo']:.2f} |
| Ábaco inferior | d'/h = {selecao_abaco['abaco_inferior']:.2f} |
| Ábaco superior | d'/h = {selecao_abaco['abaco_superior']:.2f} |
| Situação | {selecao_abaco['situacao']} |

## 4. Obtenção de ω

| Item | Valor |
|---|---|
| Modo | {resultado_omega['modo']} |
| Método | {resultado_omega['metodo']} |
| ω adotado | {resultado_omega['omega']:.4f} |
| Fonte | {fonte_omega} |

## 5. Área de aço

| Item | Valor |
|---|---:|
| As calculada pelo ábaco | {resultado_as['As_calc_cm2']:.2f} cm² |
| As mínima por Nd | {resultado_as['As_min_por_N_cm2']:.2f} cm² |
| As mínima por 0,4% Ac | {resultado_as['As_min_por_Ac_cm2']:.2f} cm² |
| As mínima adotada | {resultado_as['As_min_cm2']:.2f} cm² |
| As requerida | {resultado_as['As_req_cm2']:.2f} cm² |
| As por face | {resultado_as['As_por_face_cm2']:.2f} cm² |
| As máxima preliminar | {resultado_as['As_max_cm2']:.2f} cm² |

## 6. Momento mínimo preliminar

| Item | Valor |
|---|---:|
| e_min | {resultado_as['e_min_cm']:.2f} cm |
| Md_min | {resultado_as['Md_min_kNm']:.2f} kNm |

## 7. Sugestão preliminar de detalhamento

{sugestao_txt}

## 8. Alertas e pendências

{formatar_alertas(resultado_as['alertas'] + resultado_omega.get('alertas', []))}

## 9. Limitações

Este memorial é preliminar e não substitui a verificação estrutural completa.  
Devem ser avaliados, no mínimo: efeitos de segunda ordem, esbeltez, imperfeições geométricas,
combinações de ações, detalhamento, espaçamentos, cobrimento, ancoragens, emendas e domínio de validade dos ábacos.

## 10. Observações

{d.observacoes or "Sem observações adicionais."}
"""
    return "\n".join([linha.rstrip() for linha in md.strip().splitlines()])

memorial_md = gerar_memorial_markdown(
    dados,
    param,
    selecao_abaco,
    resultado_omega,
    resultado_as,
    df_sugestoes
)

arquivo_memorial = "memorial_flexao_composta_venturini.md"
with open(arquivo_memorial, "w", encoding="utf-8") as f:
    f.write(memorial_md)

print(f"Memorial gerado: {arquivo_memorial}")

if display and Markdown:
    display(Markdown(memorial_md))
else:
    print(memorial_md)

## 9. Evoluções recomendadas

Para transformar este notebook em ferramenta profissional de cálculo, recomenda-se:

1. manter versionamento formal do dataset digitalizado dos ábacos;
2. validar a interpolação contra exemplos resolvidos de referência;
3. implementar verificação independente por equilíbrio da seção;
4. incluir verificação de segunda ordem;
5. incluir análise de domínios de deformação;
6. gerar relatório em PDF/HTML;
7. incluir testes automatizados;
8. registrar responsável técnico e revisão do memorial;
9. documentar qualquer atualização futura da base CSV.

A principal melhoria desta versão é a incorporação do dataset saneado e validado dos ábacos, com interpolação de `ω` por lógica compatível com o uso técnico dos ábacos:

- primeiro, interpolação 2D dentro do ábaco em `(ν, μ)`;
- depois, interpolação linear entre ábacos vizinhos em `d'/h`;
- sem interpolação 3D direta como método principal;
- sem extrapolação automática fora da faixa digitalizada.
